# 02. Data Preprocessing & Feature Engineering

**Goal:** Clean the data, create new features, encode categoricals, scale numerical features, and prepare train/test splits for modeling.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
%matplotlib inline

## 2. Load Raw Data

In [2]:
df = pd.read_csv('../data/raw/telco_customer_churn.csv')
print(f"Original shape: {df.shape}")
df.head()

Original shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Basic Cleaning

In [3]:
# 3.1 Drop customerID (not useful for modeling)
df = df.drop('customerID', axis=1)

# 3.2 Convert TotalCharges to numeric (blanks become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 3.3 Check missing values after conversion
print("Missing values after conversion:")
print(df.isnull().sum())

# 3.4 Handle missing TotalCharges
# These are new customers with tenure = 0 → TotalCharges should be 0
print("\nRows with missing TotalCharges:")
print(df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 3.5 Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"\nCleaned shape: {df.shape}")
print(f"Churn distribution:\n{df['Churn'].value_counts(normalize=True).round(3)}")

Missing values after conversion:
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Rows with missing TotalCharges:
      tenure  MonthlyCharges  TotalCharges Churn
488        0           52.55           NaN    No
753        0           20.25           NaN    No
936        0           80.85           NaN    No
1082       0           25.75           NaN    No
1340       0           56.05           NaN    No
3331       0           19.85           NaN    No
3826       0           25.35           NaN    No
4380       0           20.00           NaN    No
5218       0        

## 4. Feature Engineering

Create meaningful features that can improve model performance and interpretability.

In [4]:
# 4.1 Tenure Groups (categorical)
def tenure_group(tenure):
    if tenure <= 12:
        return '0-12 months'
    elif tenure <= 24:
        return '13-24 months'
    elif tenure <= 48:
        return '25-48 months'
    else:
        return '49+ months'

df['TenureGroup'] = df['tenure'].apply(tenure_group)

# 4.2 Average Monthly Charges (handle tenure=0)
df['AvgMonthlyCharges'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

# 4.3 Number of additional services
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']

# Count how many services are active ("Yes")
df['NumServices'] = df[service_cols].apply(lambda x: (x == 'Yes').sum(), axis=1)

# 4.4 Has both Phone and Internet
df['PhoneAndInternet'] = np.where(
    (df['PhoneService'] == 'Yes') & (df['InternetService'] != 'No'), 1, 0
)

# 4.5 Senior Citizen with Dependents (interaction idea)
df['SeniorWithDependents'] = np.where(
    (df['SeniorCitizen'] == 1) & (df['Dependents'] == 'Yes'), 1, 0
)

print("New features created:")
print(df[['TenureGroup', 'AvgMonthlyCharges', 'NumServices', 'PhoneAndInternet']].head(10))

New features created:
    TenureGroup  AvgMonthlyCharges  NumServices  PhoneAndInternet
0   0-12 months          29.850000            1                 0
1  25-48 months          55.573529            2                 1
2   0-12 months          54.075000            2                 1
3  25-48 months          40.905556            3                 0
4   0-12 months          75.825000            0                 1
5   0-12 months         102.562500            3                 1
6  13-24 months          88.609091            2                 1
7   0-12 months          30.190000            1                 0
8  25-48 months         108.787500            4                 1
9    49+ months          56.257258            2                 1


In [5]:
# Quick check of new features vs Churn
print("Churn rate by TenureGroup:")
print(df.groupby('TenureGroup')['Churn'].mean().sort_values(ascending=False).round(3))

print("\nChurn rate by NumServices:")
print(df.groupby('NumServices')['Churn'].mean().round(3))

Churn rate by TenureGroup:
TenureGroup
0-12 months     0.474
13-24 months    0.287
25-48 months    0.204
49+ months      0.095
Name: Churn, dtype: float64

Churn rate by NumServices:
NumServices
0    0.214
1    0.458
2    0.358
3    0.274
4    0.223
5    0.124
6    0.053
Name: Churn, dtype: float64


## 5. Prepare Features for Modeling

In [6]:
# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (7043, 24)
Target shape: (7043,)


In [7]:
# Identify column types
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharges', 'NumServices']
binary_features = ['SeniorCitizen', 'PhoneAndInternet', 'SeniorWithDependents']  # already 0/1

categorical_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'TenureGroup'
]

print("Numerical:", numerical_features)
print("Categorical:", categorical_features)

Numerical: ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharges', 'NumServices']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup']


## 6. Encoding & Scaling Pipeline

We will use `ColumnTransformer` + `OneHotEncoder` for a clean, reproducible pipeline.

In [8]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'  # keeps binary features as-is
)

# Fit on full data first to get feature names (optional, for inspection)
X_transformed = preprocessor.fit_transform(X)

# Get feature names after one-hot encoding
ohe_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + list(ohe_feature_names) + binary_features

print(f"Transformed shape: {X_transformed.shape}")
print(f"Number of features after encoding: {len(all_feature_names)}")

Transformed shape: (7043, 37)
Number of features after encoding: 37


## 7. Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # important for imbalanced data
)

print(f"Train shape: {X_train.shape}, Churn rate: {y_train.mean():.3f}")
print(f"Test shape : {X_test.shape}, Churn rate: {y_test.mean():.3f}")

Train shape: (5634, 24), Churn rate: 0.265
Test shape : (1409, 24), Churn rate: 0.265


In [10]:
# Fit preprocessor ONLY on training data (avoid data leakage)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"Processed Train: {X_train_processed.shape}")
print(f"Processed Test : {X_test_processed.shape}")

Processed Train: (5634, 37)
Processed Test : (1409, 37)


## 8. Save Artifacts for Next Notebook

In [11]:
import os
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save processed arrays + original splits (useful for SHAP later)
np.save('../data/processed/X_train.npy', X_train_processed)
np.save('../data/processed/X_test.npy', X_test_processed)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_test.npy', y_test.values)

# Save original (unprocessed) for SHAP / interpretation
X_train.to_csv('../data/processed/X_train_raw.csv', index=False)
X_test.to_csv('../data/processed/X_test_raw.csv', index=False)

# Save preprocessor and feature names
joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump(all_feature_names, '../models/feature_names.pkl')

# Also save the full cleaned dataframe for insights notebook
df.to_csv('../data/processed/cleaned_churn_data.csv', index=False)

print("All artifacts saved successfully!")

All artifacts saved successfully!


## 9. Optional: Handle Class Imbalance (SMOTE)

Uncomment if you want to oversample the minority class **only on training data**.

In [12]:
# from imblearn.over_sampling import SMOTE

# smote = SMOTE(random_state=42)
# X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

# print(f"Before SMOTE: {np.bincount(y_train)}")
# print(f"After SMOTE : {np.bincount(y_train_smote)}")

# # Then use X_train_smote, y_train_smote for training
# np.save('../data/processed/X_train_smote.npy', X_train_smote)
# np.save('../data/processed/y_train_smote.npy', y_train_smote)

### Next Steps
→ Go to **03_model_building_evaluation.ipynb**